In [0]:
from pyspark.sql.functions import col, max as spark_max, count, countDistinct, lit, row_number
from pyspark.sql.window import Window

In [0]:
# Verificar si Ingesta_Producto encontró archivos nuevos
try:
    has_data = dbutils.jobs.taskValues.get(taskKey="Ingesta_Producto", key="has_data", debugValue="true")
    
    if has_data == "false":
        print("="*60)
        print("⚠️  INGESTA NO ENCONTRÓ ARCHIVOS NUEVOS")
        print("="*60)
        print("ℹ️  Ingesta_Producto no procesó archivos nuevos (normal para dimensiones).")
        print("ℹ️  No hay datos en Bronze para procesar a Silver.")
        print("✅ Quality_Producto finalizado exitosamente (sin procesamiento).")
        print("="*60)
        dbutils.notebook.exit("SKIPPED: No hay datos nuevos de producto")
except Exception as e:
    # Si no existe el taskValue o hay error, continuar normalmente
    print(f"ℹ️  No se pudo leer taskValue (continuando): {str(e)}")
    pass

In [0]:
# Configuración
tabla_bronze = "adbsmartdatamanuelestrada.bronze.base_producto_brz"
tabla_silver = "adbsmartdatamanuelestrada.silver.base_producto_slv"

print("="*60)
print("PIPELINE SILVER - PRODUCTOS")
print("="*60)
print(f"Tabla origen (Bronze): {tabla_bronze}")
print(f"Tabla destino (Silver): {tabla_silver}")
print("="*60)

# Leer toda la tabla bronze
df_bronze_completo = spark.table(tabla_bronze)

# Obtener el último data_ingestion_ts
ultimo_ingestion = df_bronze_completo.select(spark_max("data_ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]
print(f"\n\u2139️  Último data_ingestion_ts: {ultimo_ingestion}")

# Filtrar solo registros del último lote
df_nuevo_lote = df_bronze_completo.filter(col("data_ingestion_ts") == ultimo_ingestion)

total_registros_nuevos = df_nuevo_lote.count()
total_codigos_nuevos = df_nuevo_lote.select("cod_producto").distinct().count()

print(f"\n✅ Registros en el nuevo lote: {total_registros_nuevos:,}")
print(f"✅ Códigos únicos en el nuevo lote: {total_codigos_nuevos:,}")
print("="*60)

In [0]:
print("="*60)
print("VALIDACIÓN #0: CONTROL DE NULOS EN CAMPOS CRÍTICOS")
print("="*60)
print("Regla: Los campos críticos NO pueden ser nulos")
print("Campos críticos: cod_producto, producto")
print("-"*60)

# Definir campos críticos que no pueden ser nulos
campos_criticos = ["cod_producto", "producto"]

# Verificar nulos en cada campo crítico
nulos_encontrados = False
detalle_nulos = {}

for campo in campos_criticos:
    df_con_nulos = df_nuevo_lote.filter(col(campo).isNull())
    cantidad_nulos = df_con_nulos.count()
    
    if cantidad_nulos > 0:
        nulos_encontrados = True
        detalle_nulos[campo] = cantidad_nulos
        print(f"\n❌ ERROR: Se encontraron {cantidad_nulos:,} registros con {campo} = NULL")
        
        # Mostrar ejemplos de registros con nulos
        print(f"\n🔍 Ejemplos de registros con {campo} nulo:")
        display(df_con_nulos.limit(5))

if nulos_encontrados:
    print("\n" + "="*60)
    print("❌ PIPELINE ABORTADO POR DATOS MALFORMADOS")
    print("="*60)
    print("\n⚠️  RESUMEN DE NULOS DETECTADOS:")
    for campo, cantidad in detalle_nulos.items():
        print(f"  • {campo}: {cantidad:,} registros nulos")
    print("\n⚠️  ACCIÓN REQUERIDA:")
    print("  1. Revisar el archivo origen en la capa Landing")
    print("  2. Corregir los registros con valores nulos")
    print("  3. Volver a ejecutar la ingesta Bronze")
    print("  4. Ejecutar nuevamente este pipeline Silver")
    print("\n" + "="*60)
    
    # Abortar ejecución
    raise ValueError(
        f"VALIDACIÓN DE NULOS FALLIDA: Se encontraron {sum(detalle_nulos.values())} registros con nulos en campos críticos. "
        f"Campos afectados: {', '.join(detalle_nulos.keys())}. "
        f"Los datos deben ser corregidos en origen antes de continuar."
    )
else:
    print("\n✅ Todos los registros tienen valores válidos en campos críticos")
    print(f"✅ {total_registros_nuevos:,} registros pasaron la validación de nulos")
    print("="*60)

In [0]:
print("="*60)
print("VALIDACIÓN #1: DUPLICADOS EXACTOS DENTRO DEL LOTE")
print("="*60)
print("Definición: Registros con todas las columnas iguales excepto fecha_registro")
print("-"*60)

# Columnas para comparar (todas excepto fecha_registro y data_ingestion_ts)
columnas_comparacion = [
    "cod_producto",
    "cod_subcategoria",
    "producto",
    "color",
    "precio_catalogo",
    "tamanio",
    "rango_tamanio",
    "linea",
    "modelo"
]

# Contar duplicados exactos dentro del lote
df_duplicados_lote = df_nuevo_lote.groupBy(*columnas_comparacion).count().filter(col("count") > 1)

num_duplicados_lote = df_duplicados_lote.count()
codigos_duplicados_lote = df_duplicados_lote.select("cod_producto").distinct().count()

if num_duplicados_lote > 0:
    print(f"\n⚠️  Se encontraron {num_duplicados_lote:,} grupos de duplicados exactos en el lote")
    print(f"⚠️  Códigos de producto afectados: {codigos_duplicados_lote:,}")
    
    # Mostrar ejemplos
    print("\n🔍 Ejemplos de duplicados:")
    display(df_duplicados_lote.orderBy(col("count").desc()).limit(5))
    
    # Deduplicar: mantener solo el primer registro por grupo
    window_spec = Window.partitionBy(*columnas_comparacion).orderBy("fecha_registro")
    df_lote_sin_dup_internos = df_nuevo_lote.withColumn("row_num", row_number().over(window_spec)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")
    
    registros_eliminados_lote = total_registros_nuevos - df_lote_sin_dup_internos.count()
    print(f"\n✅ Registros eliminados por duplicación interna: {registros_eliminados_lote:,}")
else:
    print("\n✅ No se encontraron duplicados exactos dentro del lote")
    df_lote_sin_dup_internos = df_nuevo_lote
    registros_eliminados_lote = 0

print("\n" + "-"*60)
print("RESUMEN VALIDACIÓN #1:")
print("-"*60)
print(f"Total códigos en lote nuevo: {total_codigos_nuevos:,}")
print(f"Códigos duplicados en lote: {codigos_duplicados_lote:,}")
print(f"Registros que pasan a siguiente validación: {df_lote_sin_dup_internos.count():,}")
print("="*60)

In [0]:
print("="*60)
print("VALIDACIÓN #2: DUPLICADOS VS TABLA BRONZE COMPLETA")
print("="*60)
print("Definición: Registros nuevos que ya existen en la tabla (excepto fecha_registro)")
print("Regla: Si hay duplicado, prevalece el registro que ya está en la tabla")
print("-"*60)

# Separar registros antiguos (no del último lote)
df_bronze_antiguo = df_bronze_completo.filter(col("data_ingestion_ts") != ultimo_ingestion)

total_registros_antiguos = df_bronze_antiguo.count()
print(f"\nℹ️  Registros en tabla Bronze (lotes anteriores): {total_registros_antiguos:,}")

if total_registros_antiguos > 0:
    # Anti-join DIRECTO: mantener solo registros que NO existen en lotes anteriores
    # Compara SOLO por cod_producto (clave primaria)
    df_final_limpio = df_lote_sin_dup_internos.join(
        df_bronze_antiguo.select("cod_producto").distinct(),
        on="cod_producto",
        how="left_anti"
    )
    
    registros_antes = df_lote_sin_dup_internos.count()
    registros_despues = df_final_limpio.count()
    registros_eliminados_vs_tabla = registros_antes - registros_despues
    
    if registros_eliminados_vs_tabla > 0:
        print(f"\n⚠️  Se encontraron {registros_eliminados_vs_tabla:,} registros duplicados (ya existen en tabla)")
        # Calcular códigos eliminados: los que están en el lote pero NO en df_final_limpio
        codigos_eliminados = df_lote_sin_dup_internos.join(
            df_final_limpio.select("cod_producto"),
            on="cod_producto",
            how="left_anti"
        ).select("cod_producto").distinct().count()
        print(f"⚠️  Códigos de producto eliminados: {codigos_eliminados:,}")
        print(f"✅ Registros únicos que pasan a Silver: {registros_despues:,}")
    else:
        print("\n✅ No se encontraron duplicados con la tabla existente")
        registros_eliminados_vs_tabla = 0
        codigos_eliminados = 0
    
    num_duplicados_vs_tabla = registros_eliminados_vs_tabla
    codigos_duplicados_vs_tabla = codigos_eliminados
else:
    print("\nℹ️  La tabla Bronze está vacía (primera carga). Todos los registros pasan.")
    df_final_limpio = df_lote_sin_dup_internos
    num_duplicados_vs_tabla = 0
    codigos_duplicados_vs_tabla = 0
    registros_eliminados_vs_tabla = 0

registros_finales = df_final_limpio.count()
codigos_finales = df_final_limpio.select("cod_producto").distinct().count()

print("\n" + "-"*60)
print("RESUMEN VALIDACIÓN #2:")
print("-"*60)
print(f"Total códigos evaluados: {df_lote_sin_dup_internos.select('cod_producto').distinct().count():,}")
print(f"Códigos duplicados vs tabla: {codigos_duplicados_vs_tabla:,}")
print(f"Registros que pasan a Silver: {registros_finales:,}")
print(f"Códigos que pasan a Silver: {codigos_finales:,}")
print("="*60)

In [0]:
from pyspark.sql.functions import to_date, to_timestamp, round as spark_round

print("="*60)
print("VALIDACIÓN #3: FORMATO Y TIPOS DE DATOS")
print("="*60)
print("Aplicando formato correcto a columnas específicas...")
print("-"*60)

# Aplicar casting y formato correcto
# Nota: tamanio se mantiene como string (tiene valores como 'M', 'L', 'XL')
df_final_limpio = df_final_limpio \
    .withColumn("fecha_registro", to_date(col("fecha_registro"))) \
    .withColumn("data_ingestion_ts", to_timestamp(col("data_ingestion_ts"))) \
    .withColumn("precio_catalogo", spark_round(col("precio_catalogo").cast("double"), 2))

print("\n✅ Formato aplicado:")
print("  • fecha_registro → date")
print("  • data_ingestion_ts → timestamp")
print("  • precio_catalogo → decimal (2 decimales)")
print("  • tamanio → string (mantiene valores como 'M', 'L', 'XL')")

print("\nEsquema actualizado:")
df_final_limpio.select("fecha_registro", "data_ingestion_ts", "precio_catalogo", "tamanio").printSchema()

print("\nVista previa de tipos aplicados:")
display(df_final_limpio.select(
    "cod_producto",
    "producto",
    "precio_catalogo",
    "tamanio",
    "fecha_registro",
    "data_ingestion_ts"
).limit(5))

print("="*60)

In [0]:
print("="*60)
print("ESCRITURA A TABLA SILVER")
print("="*60)

if registros_finales > 0:
    # Escribir a tabla Silver
    df_final_limpio.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(tabla_silver)
    
    print(f"\n✅ Se insertaron {registros_finales:,} registros en la tabla Silver")
    print(f"✅ Códigos insertados: {codigos_finales:,}")
    
    # Verificar tabla Silver
    total_silver = spark.table(tabla_silver).count()
    print(f"\nℹ️  Total registros en tabla Silver: {total_silver:,}")
else:
    print("\n⚠️  No hay registros para insertar en Silver (todos eran duplicados)")

print("="*60)

In [0]:
print("="*60)
print("RESUMEN GENERAL DEL PIPELINE SILVER - PRODUCTOS")
print("="*60)

print(f"\n📥 ENTRADA (Bronze):")
print(f"  • Registros en nuevo lote: {total_registros_nuevos:,}")
print(f"  • Códigos únicos en nuevo lote: {total_codigos_nuevos:,}")

print(f"\n🛠️  VALIDACIONES:")
print(f"  Validación #1 (Duplicados internos del lote):")
if registros_eliminados_lote > 0:
    print(f"    ⚠️  Registros eliminados: {registros_eliminados_lote:,}")
    print(f"    ⚠️  Códigos afectados: {codigos_duplicados_lote:,}")
else:
    print(f"    ✅ Sin duplicados internos")

print(f"\n  Validación #2 (Duplicados vs tabla Bronze):")
if registros_eliminados_vs_tabla > 0:
    print(f"    ⚠️  Registros eliminados: {registros_eliminados_vs_tabla:,}")
    print(f"    ⚠️  Códigos afectados: {codigos_duplicados_vs_tabla:,}")
else:
    print(f"    ✅ Sin duplicados contra tabla existente")

print(f"\n📤 SALIDA (Silver):")
print(f"  • Registros insertados: {registros_finales:,}")
print(f"  • Códigos insertados: {codigos_finales:,}")

total_eliminados = registros_eliminados_lote + registros_eliminados_vs_tabla
porcentaje_calidad = (registros_finales / total_registros_nuevos * 100) if total_registros_nuevos > 0 else 0

print(f"\n📉 MÉTRICAS:")
print(f"  • Total registros eliminados: {total_eliminados:,}")
print(f"  • Tasa de calidad: {porcentaje_calidad:.2f}%")

print("\n" + "="*60)
print("✅ PIPELINE COMPLETADO")
print("="*60)